In [0]:
# task 3: near real-time event processing
# read clickstream events, handle bad records, compute session metrics per customer

from pyspark.sql import functions as F, Window

RAW = "/Volumes/workspace/catenaretail/raw"
EVENTS_PATH = f"{RAW}/clickstream_events.json"

# the file is json lines (one json object per line).
# mode PERMISSIVE keeps going on bad records instead of failing the job,
# and _corrupt_record captures anything unparseable.
events_raw = (spark.read
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(EVENTS_PATH))

total = events_raw.count()
print(f"total event records read: {total}")
events_raw.printSchema()
display(events_raw.limit(5))

In [0]:
# check what is actually wrong with the events before computing metrics

print("total events:", events_raw.count())
print("missing customer_id:", events_raw.filter("customer_id is null or customer_id = ''").count())
print("missing session_id:", events_raw.filter("session_id is null or session_id = ''").count())
print("missing timestamp:", events_raw.filter("timestamp is null or timestamp = ''").count())
print("missing event_type:", events_raw.filter("event_type is null or event_type = ''").count())
print("event types:", [r["event_type"] for r in events_raw.select("event_type").distinct().collect()])

In [0]:
# keep only events we can attribute to a customer session with a valid time.
# a session metric per customer needs customer_id, session_id and timestamp.

events = (events_raw
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != ""))
    .filter(F.col("session_id").isNotNull() & (F.col("session_id") != ""))
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .filter(F.col("event_ts").isNotNull()))

kept = events.count()
print(f"events kept for sessionization: {kept} (dropped {events_raw.count() - kept})")
display(events.limit(5))

In [0]:
# the raw session_ids span many days, which is not a real session.
# rebuild sessions properly: a new session starts when a customer is idle > 30 minutes.

GAP_SECONDS = 30 * 60

w = Window.partitionBy("customer_id").orderBy("event_ts")

sessionized = (events
    .withColumn("prev_ts", F.lag("event_ts").over(w))
    .withColumn("gap", F.col("event_ts").cast("long") - F.col("prev_ts").cast("long"))
    # mark the first event of a new session (no previous event, or gap too big)
    .withColumn("is_new_session", F.when(F.col("prev_ts").isNull() | (F.col("gap") > GAP_SECONDS), 1).otherwise(0))
    # cumulative sum gives each session a running number per customer
    .withColumn("session_num", F.sum("is_new_session").over(w))
    .withColumn("derived_session_id", F.concat_ws("_", "customer_id", "session_num")))

session_metrics = (sessionized
    .groupBy("customer_id", "derived_session_id")
    .agg(
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end"),
        F.count("*").alias("event_count"),
        F.sum(F.when(F.col("event_type") == "page_view", 1).otherwise(0)).alias("page_views"))
    .withColumn("duration_seconds",
        F.col("session_end").cast("long") - F.col("session_start").cast("long")))

print("total sessions (30-min gap):", session_metrics.count())
display(session_metrics.orderBy(F.col("event_count").desc()).limit(10))

In [0]:
# write session metrics as parquet for downstream analytics
OUT = "/Volumes/workspace/catenaretail/output"
sessions_out = f"{OUT}/session_metrics"

session_metrics.write.mode("overwrite").parquet(sessions_out)
print("written:", spark.read.parquet(sessions_out).count(), "sessions to", sessions_out)